# Generalised Linear Models (GLMs)

:::{tip}
The acronym for a Generalised Linear Model is most naturally GLM. However, there is scope for confusion here because the normal linear model is sometimes known as the *General Linear Model*, which is also abbreviated to GLM. So, we have two different frameworks, both of which can be referred to as GLM. To keep consistency with the function names in `R`, we will refer to *generalised* linear models as GLMs, because they are fit using the `glm()` function. Linear models are simply LMs, because they are fit using the `lm()` function. Just be aware that the term "GLM" is somewhat ambiguous. 
:::

## The Exponential Family of Distributions

The general form is given by

$$
f(y|\theta,\phi) = \exp\left[\frac{y\theta - b(\theta)}{a(\phi)} + c\left(y,\phi\right)\right],
$$

which, on the face of it, may not look particularly intuitive or useful. However, it is possible to get many different distributions to fit this definition. The usefulness of this is that aspects such as parameter estimation can be conducted identically with a variety of different distributions, rather than being restricted to a single distribution. So, as long as the chosen distribution can be made to fit the expression given above, it will work within the framework. This is the first step towards a general-purpose modelling approach that can accommodate distributions beyond the normal distribution

All distributions in the exponential family can be defined using two parameters: $\theta$ and $\phi$. These are known generally as the *location* and *scale* parameters, and can be thought of as generalisations of the *mean* and *variance*. Indeed, as we will see below, a normal distribution is defined by setting $\theta = \mu$ and $\phi = \sigma^{2}$. So, we can still conceptualise GLMs in terms of *mean functions* and *variance functions*, much like we saw last semester. 

### Specialisations of the Exponential Form
To see how the exponential form can be turned into different distributions, we will give a few examples before showing these with some actual numbers in `R` below. In all cases, do not get too hung up on the mathematics itself. The more important element is understanding that different distributions can all be represented using the same formula, and that it is this shared formula that allows for a generic modelling framework that is not restricted to just a single distribution.

### Example in `R`

In [1]:
dexp_family <- function(y,theta,phi,a,b,c){
  exp(((y*theta - b(theta))/a(phi)) + c(y,phi))
}

Now all we have to do is provide a value for `y`, `theta` and `phi`, as well as definitions of `a(phi)`, `b(theta)` and `c(y,theta)`. Depending on these choices, we can turn the `pexp_family()` function into a variety of different distribution. 

For example, we can create a normal distribution using:

In [13]:
a <- function(phi){     phi                          } 
b <- function(theta){  (theta^2)/2                   }
c <- function(y,phi){ -((y^2)/phi + log(2*pi*phi))/2 }

# Parameters of the original distribution
mu    <- 5
sigma <- 2

# Parameters of the exponential form
theta <- mu 
phi   <- sigma^2

# Example data value
y <- 7

print(dexp_family(y,theta,phi,a,b,c)) # density from exponential family
print(dnorm(y,mu,sigma))              # density of normal distribution

[1] 0.1209854
[1] 0.1209854


The value for the density of the distribution when $y = 7$ is identical whether we use the exponential form or the standard form of the normal distribution, illustrating how the same distribution can be defined both ways. We could also create a Poisson distribution using the same structure with:

In [11]:
a <- function(phi){    1                 }
b <- function(theta){  exp(theta)        }
c <- function(y,phi){ -log(factorial(y)) }

# Parameters of the original distribution
mu <- 2

# Parameters of the exponential form
theta <- log(mu)
phi   <- 1

# Data value
y <- 7

print(dexp_family(y,theta,phi,a,b,c)) # density from exponential family
print(dpois(y,mu))                    # density of Poisson distribution

[1] 0.003437087
[1] 0.003437087


Again, the densities are identical. A binomial distribution is the same. If we fix the number of trial to $n = 5$, we have:

In [10]:
a <- function(phi){   1                     }
b <- function(theta){ 5*log(1 + exp(theta)) }
c <- function(y,phi){ log(choose(5,y))      }

# Parameters of the original distribution
n <- 5
p <- 0.8

# Parameters of the exponential form
theta <- log(p/(1-p))
phi   <- 1

# Data value
y <- 3

print(dexp_family(y,theta,phi,a,b,c)) # density from exponential family
print(dbinom(y,n,p))                  # density of Binomial distribution

[1] 0.2048
[1] 0.2048


So, we can see in each case that the generic form of a distribution from the exponential family can be used to define a variety of different distributions, simply by changing a few of the parameters. Having all these distributions fall under the same mathematical formulation is very powerful because if we have a modelling framework based around the general exponential form of a distribution, we can then apply that framework generically to many different distributions. So, rather than being stuck using the normal distribution for everything, we suddenly have a choice in the assumed distribution of the data. Even more simply, we can now analyse binary, counts, proportions and ordinal outcomes, as well as continuous outcomes, opening up a much wider range of variables we can model.

## Link Functions

... Notice in the code examples above that an extra conversion was need to turn the parameters of the original distribution into a form suitable for the exponential form. For example, we used `theta <- log(mu)` for the Poisson distribution and `theta <- log(p/(1-p))` for the binomial distribution. These are both examples of *link functions*.

### Link Functions vs Data Transformations

## Estimating GLMs
... In general, these methods require the use of *maximum likelihood*. So, this is the first example we see where ordinary least squares is not longer suitable and the more general framework of maximum likelihood becomes necessary. Similarly, a Gaussian GLM is one of the few examples where closed-form solutions exist. As such, most implementations will require the use of optimisation algorithms to find the parameters that best fit. In `R`, the `glm()` function uses an optimisation technique known as *iteratively reweighted least squares* (IRLS). Remember, this is *not* least-squares, despite the name, it is an optimisation method used to find the parameters that maximise the likelihood. 